In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv", dtype={"timestamp": "int64"})
trace_df = pd.read_csv("trace.csv", dtype={"timestamp": "int64"})
log_df = pd.read_csv("log.csv", dtype={"timestamp": "int64"})
error_df = pd.read_csv("error_logs.csv", dtype={"timestamp": "int64"})

# File-level time ranges (min/max as integers)
file_ranges = {
    "metric.csv": {"min_ts": int(metric_df["timestamp"].min()) if not metric_df.empty else None,
                   "max_ts": int(metric_df["timestamp"].max()) if not metric_df.empty else None},
    "trace.csv":  {"min_ts": int(trace_df["timestamp"].min()) if not trace_df.empty else None,
                   "max_ts": int(trace_df["timestamp"].max()) if not trace_df.empty else None},
    "log.csv":    {"min_ts": int(log_df["timestamp"].min()) if not log_df.empty else None,
                   "max_ts": int(log_df["timestamp"].max()) if not log_df.empty else None},
    "error_logs.csv": {"min_ts": int(error_df["timestamp"].min()) if not error_df.empty else None,
                       "max_ts": int(error_df["timestamp"].max()) if not error_df.empty else None},
}

# Helper to aggregate pairs
def aggregate_pairs(df, id_col, name_col, top_n=20):
    if df.empty:
        return pd.DataFrame(columns=[id_col, "name", "count", "min_ts", "max_ts"])
    g = (
        df.groupby([id_col, name_col])["timestamp"]
        .agg(count="size", min_ts="min", max_ts="max")
        .reset_index()
        .rename(columns={name_col: "name"})
        .astype({"min_ts": "int64", "max_ts": "int64", "count": "int64"})
        .sort_values("count", ascending=False)
    )
    return g[[id_col, "name", "count", "min_ts", "max_ts"]].head(top_n)

metric_pairs_df = aggregate_pairs(metric_df, "cmdb_id", "kpi_name", top_n=20)
trace_pairs_df  = aggregate_pairs(trace_df,  "cmdb_id", "trace_name", top_n=20)
log_pairs_df    = aggregate_pairs(log_df,    "cmdb_id", "log_name", top_n=20)

# error_logs summary: total count and top 10 cmdb_id counts
total_error_count = int(len(error_df))
if error_df.empty:
    top_error_cmdb = pd.DataFrame(columns=["cmdb_id", "count"])
else:
    top_error_cmdb = (
        error_df.groupby("cmdb_id")["timestamp"]
        .agg(count="size")
        .reset_index()
        .astype({"count": "int64"})
        .sort_values("count", ascending=False)
        .head(10)
    )

# Build JSON-serializable result
result = {
    "file_ranges": file_ranges,
    "metric_pairs": metric_pairs_df.to_dict(orient="records"),
    "trace_pairs": trace_pairs_df.to_dict(orient="records"),
    "log_pairs": log_pairs_df.to_dict(orient="records"),
    "error_log_summary": {
        "total_count": total_error_count,
        "top_cmdb_ids": top_error_cmdb.to_dict(orient="records"),
    },
}

result

```
Out[1]:
```
summary = (
    "Summary of telemetry (plain English):\n"
    "- Time ranges: metric.csv, trace.csv, and log.csv all span the same interval: "
    "min_ts=1706101680, max_ts=1706103120. error_logs.csv has no rows.\n"
    "- metric.csv: many (cmdb_id, kpi_name) pairs have full coverage (25 rows each). "
    "Example top pair: ('ts-voucher-service', 'socket') with 25 rows. Several services "
    "(e.g., ts-admin-basic-info-service, ts-travel2-service, ts-travel-service) report "
    "cpu, mem, diskio, latency-50/90, socket, workload, all with 25 samples.\n"
    "- trace.csv: trace pairs also have 25 rows each. Example top pair: "
    "('root', 'trace.to_ts-auth-service.row_count') with 25 rows. ts-travel2-service has "
    "multiple traces to ts-order, ts-route, ts-train (all 25 rows).\n"
    "- log.csv: many services report log.row_count and log.error_count with 25 rows. "
    "Notably, ts-auth-service and ts-basic-service appear with 25 samples. A few mongo "
    "entries (ts-auth-mongo, ts-config-mongo) have 24 rows (slightly shorter coverage).\n"
    "- error_logs.csv: total_count = 0, no error log cmdb_ids to inspect.\n\n"
    "Conclusion: metric, trace, and log telemetry are present and consistent over the same "
    "time window; there are no error log entries to analyze further."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

{'file_ranges': {'metric.csv': {'min_ts': 1706101680, 'max_ts': 1706103120}, 'trace.csv': {'min_ts': 1706101680, 'max_ts': 1706103120}, 'log.csv': {'min_ts': 1706101680, 'max_ts': 1706103120}, 'error_logs.csv': {'min_ts': None, 'max_ts': None}}, 'metric_pairs': [{'cmdb_id': 'ts-voucher-service', 'name': 'socket', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-admin-basic-info-service', 'name': 'cpu', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-admin-basic-info-service', 'name': 'diskio', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-admin-basic-info-service', 'name': 'latency-50', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-admin-basic-info-service', 'name': 'latency-90', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-admin-basic-info-service', 'name': 'mem', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel2-mongo', 'name': 'mem', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel2-mongo', 'name': 'socket', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel2-service', 'name': 'cpu', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel2-service', 'name': 'diskio', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel2-service', 'name': 'latency-50', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel2-service', 'name': 'latency-90', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel2-service', 'name': 'mem', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel2-service', 'name': 'socket', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel-service', 'name': 'diskio', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel-service', 'name': 'latency-50', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel-service', 'name': 'latency-90', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel-service', 'name': 'mem', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel-service', 'name': 'socket', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel-service', 'name': 'workload', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}], 'trace_pairs': [{'cmdb_id': 'root', 'name': 'trace.to_ts-auth-service.row_count', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel2-service', 'name': 'trace.to_ts-order-other-service.duration_mean', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel2-service', 'name': 'trace.to_ts-order-other-service.duration_p95', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel2-service', 'name': 'trace.to_ts-order-other-service.error_rate', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel2-service', 'name': 'trace.to_ts-order-other-service.row_count', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel2-service', 'name': 'trace.to_ts-route-service.duration_mean', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel2-service', 'name': 'trace.to_ts-route-service.duration_p95', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel2-service', 'name': 'trace.to_ts-route-service.error_rate', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel2-service', 'name': 'trace.to_ts-train-service.duration_mean', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel2-service', 'name': 'trace.to_ts-train-service.duration_p95', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel2-service', 'name': 'trace.to_ts-train-service.error_rate', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel2-service', 'name': 'trace.to_ts-train-service.row_count', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-price-service', 'name': 'trace.from_ts-basic-service.duration_mean', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-price-service', 'name': 'trace.from_ts-basic-service.duration_p95', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-price-service', 'name': 'trace.from_ts-basic-service.error_rate', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-price-service', 'name': 'trace.from_ts-basic-service.row_count', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-price-service', 'name': 'trace.to_ts-price-service.duration_mean', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-price-service', 'name': 'trace.to_ts-price-service.duration_p95', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-price-service', 'name': 'trace.to_ts-price-service.error_rate', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-price-service', 'name': 'trace.to_ts-price-service.row_count', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}], 'log_pairs': [{'cmdb_id': 'ts-auth-service', 'name': 'log.row_count', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-auth-service', 'name': 'log.error_count', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-basic-service', 'name': 'log.row_count', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-basic-service', 'name': 'log.error_count', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel2-service', 'name': 'log.row_count', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-travel2-service', 'name': 'log.error_count', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-route-service', 'name': 'log.error_count', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-route-service', 'name': 'log.row_count', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-seat-service', 'name': 'log.row_count', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-seat-service', 'name': 'log.error_count', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-order-other-service', 'name': 'log.row_count', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-order-other-service', 'name': 'log.error_count', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-price-service', 'name': 'log.row_count', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-price-service', 'name': 'log.error_count', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-ui-dashboard', 'name': 'log.row_count', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-ui-dashboard', 'name': 'log.error_count', 'count': 25, 'min_ts': 1706101680, 'max_ts': 1706103120}, {'cmdb_id': 'ts-auth-mongo', 'name': 'log.error_count', 'count': 24, 'min_ts': 1706101680, 'max_ts': 1706103060}, {'cmdb_id': 'ts-auth-mongo', 'name': 'log.row_count', 'count': 24, 'min_ts': 1706101680, 'max_ts': 1706103060}, {'cmdb_id': 'ts-config-mongo', 'name': 'log.error_count', 'count': 24, 'min_ts': 1706101680, 'max_ts': 1706103060}, {'cmdb_id': 'ts-config-mongo', 'name': 'log.row_count', 'count': 24, 'min_ts': 1706101680, 'max_ts': 1706103060}], 'error_log_summary': {'total_count': 0, 'top_cmdb_ids': []}}```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse metric_df from previous state
df = metric_df.copy()

candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Filter to candidate services for threshold computation (use full series for thresholds)
df_cand = df[df["cmdb_id"].isin(candidates)].copy()

# Compute global thresholds (P95, P05) and total sample count per (cmdb_id, kpi_name)
if not df_cand.empty:
    thresholds_df = (
        df_cand.groupby(["cmdb_id", "kpi_name"])["value"]
        .agg(total_count="size",
             global_p95=lambda s: float(s.quantile(0.95)),
             global_p05=lambda s: float(s.quantile(0.05)))
        .reset_index()
    )
else:
    thresholds_df = pd.DataFrame(columns=["cmdb_id","kpi_name","total_count","global_p95","global_p05"])

# Parse timestamps to UTC datetimes for filtering
df_cand["ts_utc"] = pd.to_datetime(df_cand["timestamp"], unit="s", utc=True)

# Define incident window in UTC
start = pd.to_datetime("2024-01-24 13:05:01", utc=True)
end   = pd.to_datetime("2024-01-24 13:35:01", utc=True)

# Filter to window
window_df = df_cand[(df_cand["ts_utc"] >= start) & (df_cand["ts_utc"] <= end)].copy()

# Prepare empty result structures
anomaly_summary_df = pd.DataFrame(columns=[
    "cmdb_id","kpi_name","total_count","global_p95","global_p05",
    "anomaly_count_in_window","earliest_anomaly_ts","max_value_in_window","min_value_in_window","severity"
])

# If there is data in the window and thresholds computed, analyze anomalies
if (not window_df.empty) and (not thresholds_df.empty):
    # Join thresholds to each window row
    window_with_thresh = window_df.merge(thresholds_df, on=["cmdb_id","kpi_name"], how="left", validate="m:1")
    # Determine anomaly condition per row
    window_with_thresh["is_high_anom"] = window_with_thresh["value"] >= window_with_thresh["global_p95"]
    window_with_thresh["is_low_anom"]  = window_with_thresh["value"] <= window_with_thresh["global_p05"]
    window_with_thresh["is_anom"] = window_with_thresh["is_high_anom"] | window_with_thresh["is_low_anom"]

    # Group and summarize using apply to allow conditional earliest anomaly timestamp
    def summarize_group(g):
        anomaly_count = int(g["is_anom"].sum())
        earliest = int(g.loc[g["is_anom"], "timestamp"].min()) if anomaly_count > 0 else None
        return pd.Series({
            "anomaly_count_in_window": anomaly_count,
            "earliest_anomaly_ts": earliest,
            "max_value_in_window": float(g["value"].max()) if not g["value"].isna().all() else None,
            "min_value_in_window": float(g["value"].min()) if not g["value"].isna().all() else None,
        })

    grp_summary = window_with_thresh.groupby(["cmdb_id","kpi_name"]).apply(summarize_group).reset_index()

    # Merge with global thresholds and total_count
    anomaly_summary_df = grp_summary.merge(thresholds_df, on=["cmdb_id","kpi_name"], how="left", validate="1:1")

    # Reorder columns
    anomaly_summary_df = anomaly_summary_df[
        ["cmdb_id","kpi_name","total_count","global_p95","global_p05",
         "anomaly_count_in_window","earliest_anomaly_ts","max_value_in_window","min_value_in_window"]
    ].copy()

    # Compute severity per row
    def compute_severity(row):
        gp95 = row["global_p95"]
        gp05 = row["global_p05"]
        max_w = row["max_value_in_window"]
        min_w = row["min_value_in_window"]

        sev_high = -np.inf
        sev_low = -np.inf

        # high-side severity
        if gp95 is not None and not pd.isna(gp95):
            if gp95 != 0 and max_w is not None:
                sev_high = float(max_w / gp95)
            elif gp95 == 0 and max_w is not None and max_w > 0:
                sev_high = np.inf

        # low-side severity
        if gp05 is not None and not pd.isna(gp05):
            if (min_w is not None) and (min_w != 0):
                sev_low = float(gp05 / min_w)
            elif (min_w is not None) and (min_w == 0) and gp05 > 0:
                sev_low = np.inf

        if sev_high == -np.inf and sev_low == -np.inf:
            return np.nan
        return float(max(sev_high, sev_low))

    anomaly_summary_df["severity"] = anomaly_summary_df.apply(compute_severity, axis=1)

    # Keep only pairs with anomaly_count_in_window > 0
    anomaly_summary_df = anomaly_summary_df[anomaly_summary_df["anomaly_count_in_window"] > 0].copy()

    # Sort by severity desc and limit to top 20
    anomaly_summary_df = anomaly_summary_df.sort_values("severity", ascending=False).head(20).reset_index(drop=True)

    # Ensure types are JSON-serializable (convert numpy types to native Python)
    int_cols = ["total_count","anomaly_count_in_window"]
    for c in int_cols:
        if c in anomaly_summary_df:
            anomaly_summary_df[c] = anomaly_summary_df[c].apply(lambda x: int(x) if pd.notna(x) else None)
    float_cols = ["global_p95","global_p05","max_value_in_window","min_value_in_window","severity"]
    for c in float_cols:
        if c in anomaly_summary_df:
            anomaly_summary_df[c] = anomaly_summary_df[c].apply(lambda x: float(x) if pd.notna(x) else None)
    # earliest_anomaly_ts already None or int

# Prepare thresholds output (for all scanned pairs)
thresholds_out_df = thresholds_df.copy()
if not thresholds_out_df.empty:
    thresholds_out_df = thresholds_out_df[["cmdb_id","kpi_name","total_count","global_p95","global_p05"]]
    # convert types to native Python
    thresholds_out_df["total_count"] = thresholds_out_df["total_count"].astype(int)
    thresholds_out_df["global_p95"] = thresholds_out_df["global_p95"].astype(float)
    thresholds_out_df["global_p05"] = thresholds_out_df["global_p05"].astype(float)

# Summary counts
total_pairs_scanned = int(thresholds_out_df.shape[0])
anomalous_pairs = int(anomaly_summary_df.shape[0])

result = {
    "thresholds": thresholds_out_df.to_dict(orient="records"),
    "anomaly_summary": anomaly_summary_df.to_dict(orient="records"),
    "summary_counts": {
        "total_pairs_scanned": total_pairs_scanned,
        "anomalous_pairs": anomalous_pairs
    }
}

result

```
Out[2]:
```
```python
import pandas as pd

# Reuse the previous result variable produced by the analysis step
res = result

anoms = res.get("anomaly_summary", [])
scanned = res.get("summary_counts", {}).get("total_pairs_scanned", 0)
flagged = res.get("summary_counts", {}).get("anomalous_pairs", 0)

# Prepare top 5 anomalies (already sorted by severity desc in the result)
top5 = anoms[:5]

lines = []
lines.append(f"Summary (plain English):")
lines.append(f"- Metric pairs scanned for the 5 candidate services: {scanned}")
lines.append(f"- Pairs flagged as anomalous in the incident window: {flagged}")
lines.append("- Top anomalies (by severity):")

def ts_to_utc(ts):
    try:
        return pd.to_datetime(int(ts), unit="s", utc=True).strftime("%Y-%m-%d %H:%M:%S %Z")
    except Exception:
        return str(ts)

for i, a in enumerate(top5, 1):
    cmdb = a["cmdb_id"]
    kpi = a["kpi_name"]
    sev = a.get("severity")
    maxv = a.get("max_value_in_window")
    gp95 = a.get("global_p95")
    earliest = ts_to_utc(a.get("earliest_anomaly_ts"))
    lines.append(
        f"  {i}. {cmdb} / {kpi} — severity={sev:.3f}, max_in_window={maxv}, global_p95={gp95}, earliest_anomaly={earliest}"
    )

lines.append("")
lines.append("Notes:")
lines.append("- Global P95/P05 thresholds were computed using the full available series before filtering to the incident window.")
lines.append("- An anomaly is any value >= P95 (high-side) or <= P05 (low-side).")
lines.append("- All reported fields are taken from the computed anomaly summary for the incident window 2024-01-24 13:05:01 to 2024-01-24 13:35:01 UTC.")

summary_text = "\n".join(lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

{'thresholds': [{'cmdb_id': 'ts-auth-service', 'kpi_name': 'cpu', 'total_count': 25, 'global_p95': 16.912082757058357, 'global_p05': 13.209944082072168}, {'cmdb_id': 'ts-auth-service', 'kpi_name': 'diskio', 'total_count': 25, 'global_p95': 58655.80570299119, 'global_p05': 44306.20650589121}, {'cmdb_id': 'ts-auth-service', 'kpi_name': 'latency-50', 'total_count': 25, 'global_p95': 0.4239535428167591, 'global_p05': 0.22670988681654636}, {'cmdb_id': 'ts-auth-service', 'kpi_name': 'latency-90', 'total_count': 25, 'global_p95': 0.9444489779225623, 'global_p05': 0.6185855664488017}, {'cmdb_id': 'ts-auth-service', 'kpi_name': 'mem', 'total_count': 25, 'global_p95': 239694929.92000002, 'global_p05': 224188948.48}, {'cmdb_id': 'ts-auth-service', 'kpi_name': 'socket', 'total_count': 25, 'global_p95': 14.669999999999998, 'global_p05': 12.536666666666667}, {'cmdb_id': 'ts-auth-service', 'kpi_name': 'workload', 'total_count': 25, 'global_p95': 2.8755166666666665, 'global_p05': 2.48129}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'cpu', 'total_count': 25, 'global_p95': 11.509403377910825, 'global_p05': 1.7897397175151308}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'diskio', 'total_count': 25, 'global_p95': 63123.94938335049, 'global_p05': 44891.88618623459}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'latency-50', 'total_count': 25, 'global_p95': 0.016155949464206835, 'global_p05': 0.0074064922973358}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'latency-90', 'total_count': 25, 'global_p95': 0.06396117372944751, 'global_p05': 0.01061239233024834}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'mem', 'total_count': 25, 'global_p95': 261157259.9466667, 'global_p05': 250473417.38666666}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'socket', 'total_count': 25, 'global_p95': 11.546666666666667, 'global_p05': 9.906666666666666}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'workload', 'total_count': 25, 'global_p95': 7.472633333333333, 'global_p05': 4.202896666666667}, {'cmdb_id': 'ts-route-service', 'kpi_name': 'cpu', 'total_count': 25, 'global_p95': 17.485920729566438, 'global_p05': 4.338448538867416}, {'cmdb_id': 'ts-route-service', 'kpi_name': 'diskio', 'total_count': 25, 'global_p95': 326656.7155166994, 'global_p05': 41774.6722941034}, {'cmdb_id': 'ts-route-service', 'kpi_name': 'latency-50', 'total_count': 25, 'global_p95': 0.007591548300421277, 'global_p05': 0.00371327908447054}, {'cmdb_id': 'ts-route-service', 'kpi_name': 'latency-90', 'total_count': 25, 'global_p95': 0.1424614036381787, 'global_p05': 0.00848650089873232}, {'cmdb_id': 'ts-route-service', 'kpi_name': 'mem', 'total_count': 25, 'global_p95': 262219762.34666666, 'global_p05': 258195210.23999998}, {'cmdb_id': 'ts-route-service', 'kpi_name': 'socket', 'total_count': 25, 'global_p95': 15.313333333333333, 'global_p05': 12.18}, {'cmdb_id': 'ts-route-service', 'kpi_name': 'workload', 'total_count': 25, 'global_p95': 34.79334333333333, 'global_p05': 30.584439999999997}, {'cmdb_id': 'ts-train-service', 'kpi_name': 'cpu', 'total_count': 25, 'global_p95': 12.640933950754672, 'global_p05': 2.658587907130413}, {'cmdb_id': 'ts-train-service', 'kpi_name': 'diskio', 'total_count': 25, 'global_p95': 60592.907268183524, 'global_p05': 41714.648661449035}, {'cmdb_id': 'ts-train-service', 'kpi_name': 'latency-50', 'total_count': 25, 'global_p95': 0.007683982828437719, 'global_p05': 0.0036903201948518998}, {'cmdb_id': 'ts-train-service', 'kpi_name': 'latency-90', 'total_count': 25, 'global_p95': 0.019842491488563846, 'global_p05': 0.008306483723555441}, {'cmdb_id': 'ts-train-service', 'kpi_name': 'mem', 'total_count': 25, 'global_p95': 252112896.0, 'global_p05': 247111133.86666664}, {'cmdb_id': 'ts-train-service', 'kpi_name': 'socket', 'total_count': 25, 'global_p95': 15.460790960451977, 'global_p05': 13.573333333333334}, {'cmdb_id': 'ts-train-service', 'kpi_name': 'workload', 'total_count': 25, 'global_p95': 22.038989999999995, 'global_p05': 18.453221694915253}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'cpu', 'total_count': 25, 'global_p95': 15.966915459260221, 'global_p05': 2.951554017961891}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'diskio', 'total_count': 25, 'global_p95': 69885.12024707008, 'global_p05': 42142.67258635661}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'latency-50', 'total_count': 25, 'global_p95': 0.055434613084526116, 'global_p05': 0.018744041948391223}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'latency-90', 'total_count': 25, 'global_p95': 0.5520232218307343, 'global_p05': 0.12426666331264354}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'mem', 'total_count': 25, 'global_p95': 269147163.3066667, 'global_p05': 254741913.6}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'socket', 'total_count': 25, 'global_p95': 19.22, 'global_p05': 14.166666666666668}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'workload', 'total_count': 25, 'global_p95': 11.030923333333334, 'global_p05': 6.37008}], 'anomaly_summary': [{'cmdb_id': 'ts-route-service', 'kpi_name': 'diskio', 'total_count': 25, 'global_p95': 326656.7155166994, 'global_p05': 41774.6722941034, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680.0, 'max_value_in_window': 17095565.3197128, 'min_value_in_window': 37152.76990407946, 'severity': 52.33495748792844}, {'cmdb_id': 'ts-train-service', 'kpi_name': 'latency-90', 'total_count': 25, 'global_p95': 0.019842491488563846, 'global_p05': 0.008306483723555441, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680.0, 'max_value_in_window': 0.0550410137100756, 'min_value_in_window': 0.0082174015641855, 'severity': 2.7738962993532494}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'latency-50', 'total_count': 25, 'global_p95': 0.055434613084526116, 'global_p05': 0.018744041948391223, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680.0, 'max_value_in_window': 0.1326181297257681, 'min_value_in_window': 0.0181989247311827, 'severity': 2.3923343619906463}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'latency-90', 'total_count': 25, 'global_p95': 0.06396117372944751, 'global_p05': 0.01061239233024834, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680.0, 'max_value_in_window': 0.1156590469626223, 'min_value_in_window': 0.0100606965057024, 'severity': 1.8082696144985417}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'cpu', 'total_count': 25, 'global_p95': 15.966915459260221, 'global_p05': 2.951554017961891, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680.0, 'max_value_in_window': 27.549142486860283, 'min_value_in_window': 2.7467992597191317, 'severity': 1.7253891371287244}, {'cmdb_id': 'ts-route-service', 'kpi_name': 'latency-50', 'total_count': 25, 'global_p95': 0.007591548300421277, 'global_p05': 0.00371327908447054, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680.0, 'max_value_in_window': 0.0126292247487497, 'min_value_in_window': 0.0035335629657099, 'severity': 1.6635901200877385}, {'cmdb_id': 'ts-train-service', 'kpi_name': 'latency-50', 'total_count': 25, 'global_p95': 0.007683982828437719, 'global_p05': 0.0036903201948518998, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680.0, 'max_value_in_window': 0.0117865603488231, 'min_value_in_window': 0.0036777967646388, 'severity': 1.5339128954325765}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'latency-50', 'total_count': 25, 'global_p95': 0.016155949464206835, 'global_p05': 0.0074064922973358, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680.0, 'max_value_in_window': 0.0236873665599216, 'min_value_in_window': 0.0073133915181113, 'severity': 1.4661698844998532}, {'cmdb_id': 'ts-train-service', 'kpi_name': 'cpu', 'total_count': 25, 'global_p95': 12.640933950754672, 'global_p05': 2.658587907130413, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680.0, 'max_value_in_window': 17.447616200070023, 'min_value_in_window': 2.5461210172633995, 'severity': 1.3802473984945067}, {'cmdb_id': 'ts-train-service', 'kpi_name': 'diskio', 'total_count': 25, 'global_p95': 60592.907268183524, 'global_p05': 41714.648661449035, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706102700.0, 'max_value_in_window': 63575.819265190206, 'min_value_in_window': 30416.266949551005, 'severity': 1.3714585268020476}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'latency-90', 'total_count': 25, 'global_p95': 0.5520232218307343, 'global_p05': 0.12426666331264354, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680.0, 'max_value_in_window': 0.7398383030105131, 'min_value_in_window': 0.1177645211603767, 'severity': 1.3402303992881086}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'cpu', 'total_count': 25, 'global_p95': 11.509403377910825, 'global_p05': 1.7897397175151308, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680.0, 'max_value_in_window': 13.155217428089513, 'min_value_in_window': 1.3369042646995375, 'severity': 1.338719431729366}, {'cmdb_id': 'ts-auth-service', 'kpi_name': 'diskio', 'total_count': 25, 'global_p95': 58655.80570299119, 'global_p05': 44306.20650589121, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101800.0, 'max_value_in_window': 72927.81101518922, 'min_value_in_window': 33754.99608554123, 'severity': 1.3125821846819747}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'diskio', 'total_count': 25, 'global_p95': 63123.94938335049, 'global_p05': 44891.88618623459, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680.0, 'max_value_in_window': 74677.26093468955, 'min_value_in_window': 40711.15443796223, 'severity': 1.1830258034264622}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'workload', 'total_count': 25, 'global_p95': 11.030923333333334, 'global_p05': 6.37008, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101980.0, 'max_value_in_window': 12.931133333333332, 'min_value_in_window': 5.752133333333333, 'severity': 1.1722620983374916}, {'cmdb_id': 'ts-auth-service', 'kpi_name': 'latency-90', 'total_count': 25, 'global_p95': 0.9444489779225623, 'global_p05': 0.6185855664488017, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680.0, 'max_value_in_window': 1.074424715012, 'min_value_in_window': 0.5331163024288026, 'severity': 1.1603201095719886}, {'cmdb_id': 'ts-auth-service', 'kpi_name': 'socket', 'total_count': 25, 'global_p95': 14.669999999999998, 'global_p05': 12.536666666666667, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706102100.0, 'max_value_in_window': 17.0, 'min_value_in_window': 11.85, 'severity': 1.1588275391956375}, {'cmdb_id': 'ts-route-service', 'kpi_name': 'cpu', 'total_count': 25, 'global_p95': 17.485920729566438, 'global_p05': 4.338448538867416, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680.0, 'max_value_in_window': 20.14350111848939, 'min_value_in_window': 4.104234233771634, 'severity': 1.1519840121675335}, {'cmdb_id': 'ts-auth-service', 'kpi_name': 'workload', 'total_count': 25, 'global_p95': 2.8755166666666665, 'global_p05': 2.48129, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680.0, 'max_value_in_window': 2.898883333333333, 'min_value_in_window': 2.1741525423728816, 'severity': 1.1412676671214188}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'workload', 'total_count': 25, 'global_p95': 7.472633333333333, 'global_p05': 4.202896666666667, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101980.0, 'max_value_in_window': 8.43455, 'min_value_in_window': 3.8334166666666665, 'severity': 1.1287252597254873}], 'summary_counts': {'total_pairs_scanned': 35, 'anomalous_pairs': 20}}```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse trace_df loaded previously
df = trace_df.copy()

services = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service","root"]

# 1) Global thresholds for each (cmdb_id, trace_name) pair using full series
thresholds_all = (
    df.groupby(["cmdb_id", "trace_name"])["value"]
    .agg(total_count="size",
         global_p95=lambda s: float(s.quantile(0.95)),
         global_p05=lambda s: float(s.quantile(0.05)))
    .reset_index()
)

# limit thresholds output to top 50 pairs by total_count (as requested)
trace_thresholds = thresholds_all.sort_values("total_count", ascending=False).head(50).reset_index(drop=True)

# 2) Filter trace.csv to incident window (UTC)
df["ts_utc"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
start = pd.to_datetime("2024-01-24 13:05:01", utc=True)
end   = pd.to_datetime("2024-01-24 13:35:01", utc=True)
window_df = df[(df["ts_utc"] >= start) & (df["ts_utc"] <= end)].copy()

# Keep only rows where cmdb_id or trace_name contains any of the listed service names
pattern = "|".join([s.replace("-", r"\-") for s in services])  # escape hyphens for regex safety
mask = window_df["cmdb_id"].astype(str).str.contains(pattern, regex=True) | window_df["trace_name"].astype(str).str.contains(pattern, regex=True)
window_filtered = window_df[mask].copy()

# Determine unique pairs scanned for the listed services (from full thresholds_all)
pairs_mask = thresholds_all["cmdb_id"].astype(str).str.contains(pattern, regex=True) | thresholds_all["trace_name"].astype(str).str.contains(pattern, regex=True)
total_pairs_scanned = int(thresholds_all[pairs_mask].shape[0])

# If there are rows in the filtered window, compute anomalies
if not window_filtered.empty:
    # Merge global thresholds (from thresholds_all) into window rows
    window_with_thresh = window_filtered.merge(
        thresholds_all, on=["cmdb_id", "trace_name"], how="left", validate="m:1"
    )

    # Mark anomalies using global thresholds computed from full series
    window_with_thresh["is_high_anom"] = window_with_thresh["value"] >= window_with_thresh["global_p95"]
    window_with_thresh["is_low_anom"]  = window_with_thresh["value"] <= window_with_thresh["global_p05"]
    window_with_thresh["is_anom"] = window_with_thresh["is_high_anom"] | window_with_thresh["is_low_anom"]

    # Group and summarize per pair
    def summarize(g):
        anom_count = int(g["is_anom"].sum())
        earliest = int(g.loc[g["is_anom"], "timestamp"].min()) if anom_count > 0 else None
        return pd.Series({
            "anomaly_count_in_window": anom_count,
            "earliest_anomaly_ts": earliest,
            "max_value_in_window": float(g["value"].max()) if not g["value"].isna().all() else None,
            "min_value_in_window": float(g["value"].min()) if not g["value"].isna().all() else None,
        })

    grp = window_with_thresh.groupby(["cmdb_id", "trace_name"]).apply(summarize).reset_index()

    # Merge with thresholds to include global_p95/p05 and total_count
    trace_anom = grp.merge(thresholds_all, left_on=["cmdb_id", "trace_name"], right_on=["cmdb_id", "trace_name"], how="left", validate="1:1")

    # Compute severity per rules
    def compute_severity(row):
        gp95 = row["global_p95"]
        gp05 = row["global_p05"]
        max_w = row["max_value_in_window"]
        min_w = row["min_value_in_window"]

        sev_high = -np.inf
        sev_low = -np.inf

        # high-side
        if gp95 is not None and not pd.isna(gp95):
            if gp95 != 0 and max_w is not None:
                sev_high = float(max_w / gp95)
            elif gp95 == 0 and max_w is not None and max_w > 0:
                sev_high = np.inf

        # low-side
        if gp05 is not None and not pd.isna(gp05):
            if (min_w is not None) and (min_w != 0):
                sev_low = float(gp05 / min_w)
            elif (min_w is not None) and (min_w == 0) and gp05 > 0:
                sev_low = np.inf

        if sev_high == -np.inf and sev_low == -np.inf:
            return np.nan
        return float(max(sev_high, sev_low))

    trace_anom["severity"] = trace_anom.apply(compute_severity, axis=1)

    # Keep only pairs with anomaly_count_in_window > 0
    trace_anom = trace_anom[trace_anom["anomaly_count_in_window"] > 0].copy()

    # Select and order requested columns
    trace_anom = trace_anom[[
        "cmdb_id", "trace_name", "total_count", "global_p95", "global_p05",
        "anomaly_count_in_window", "earliest_anomaly_ts", "max_value_in_window", "min_value_in_window", "severity"
    ]]

    # Sort by severity desc and limit top 20
    trace_anom = trace_anom.sort_values("severity", ascending=False).head(20).reset_index(drop=True)

    # Ensure native Python types for JSON-serializable output
    int_cols = ["total_count", "anomaly_count_in_window", "earliest_anomaly_ts"]
    for c in int_cols:
        if c in trace_anom:
            trace_anom[c] = trace_anom[c].apply(lambda x: int(x) if pd.notna(x) else None)
    float_cols = ["global_p95", "global_p05", "max_value_in_window", "min_value_in_window", "severity"]
    for c in float_cols:
        if c in trace_anom:
            trace_anom[c] = trace_anom[c].apply(lambda x: float(x) if pd.notna(x) else None)

    anomalous_pairs = int(trace_anom.shape[0])
else:
    # No data in window or no rows matching services
    trace_anom = pd.DataFrame(columns=[
        "cmdb_id", "trace_name", "total_count", "global_p95", "global_p05",
        "anomaly_count_in_window", "earliest_anomaly_ts", "max_value_in_window", "min_value_in_window", "severity"
    ])
    anomalous_pairs = 0

# Prepare trace_thresholds output (keep requested fields only)
trace_thresholds_out = trace_thresholds[["cmdb_id", "trace_name", "total_count", "global_p95", "global_p05"]].copy()
# convert types for JSON serialization
trace_thresholds_out["total_count"] = trace_thresholds_out["total_count"].astype(int)
trace_thresholds_out["global_p95"] = trace_thresholds_out["global_p95"].astype(float)
trace_thresholds_out["global_p05"] = trace_thresholds_out["global_p05"].astype(float)

# Build final result dict
result = {
    "trace_thresholds": trace_thresholds_out.to_dict(orient="records"),
    "trace_anomaly_summary": trace_anom.to_dict(orient="records"),
    "summary_counts": {
        "total_trace_pairs_scanned": total_pairs_scanned,
        "anomalous_pairs": anomalous_pairs
    }
}

result

```
Out[3]:
```
summary = (
    "Trace analysis summary (plain English):\n"
    "- Analysis window: 2024-01-24 13:05:01 to 2024-01-24 13:35:01 UTC.\n"
    "- Global P95/P05 thresholds were computed using the full available trace series before window filtering.\n"
    "- Total (cmdb_id, trace_name) pairs scanned for the listed services: 328.\n"
    "- Pairs flagged as anomalous in the window: 20.\n"
    "- Top anomalies by severity (examples):\n"
    "  1) ts-basic-service / trace.to_ts-route-service.row_count — severity=55.3 (max=160.0 vs global_p95=147.4)\n"
    "  2) ts-basic-service / trace.to_ts-train-service.row_count — severity=55.3 (max=160.0 vs global_p95=147.4)\n"
    "  3) ts-train-service / trace.from_ts-basic-service.row_count — severity=55.3 (max=160.0 vs global_p95=147.4)\n"
    "  4) ts-route-service / trace.from_ts-basic-service.row_count — severity=55.3 (max=160.0 vs global_p95=147.4)\n"
    "  5) ts-train-service / trace.to_ts-train-service.row_count — severity=53.3 (max=2848.0 vs global_p95=2644.8)\n"
    "- Many of the top anomalies are for trace 'row_count' metrics (inter-service request counts) showing large deviations above their global P95; several services (basic, route, train, auth, travel2, root) show strong row_count anomalies.\n"
    "- Interpretation: during the incident window there were pronounced spikes (high-side anomalies) in trace row_count for multiple inter-service calls. These are the most severe trace-level signals to investigate further.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

{'trace_thresholds': [{'cmdb_id': 'root', 'trace_name': 'trace.to_ts-auth-service.row_count', 'total_count': 25, 'global_p95': 85.6, 'global_p05': 65.2}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-order-other-service.duration_mean', 'total_count': 25, 'global_p95': 0.009414130526315758, 'global_p05': 0.0046344394542166}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-order-other-service.duration_p95', 'total_count': 25, 'global_p95': 0.0242832099999999, 'global_p05': 0.00529925999999998}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-order-other-service.error_rate', 'total_count': 25, 'global_p95': 0.0, 'global_p05': 0.0}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-order-other-service.row_count', 'total_count': 25, 'global_p95': 80.6, 'global_p05': 62.2}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-route-service.duration_mean', 'total_count': 25, 'global_p95': 0.02985234640397067, 'global_p05': 0.00373109139109882}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-route-service.duration_p95', 'total_count': 25, 'global_p95': 0.2096894, 'global_p05': 0.00432343999999992}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-route-service.error_rate', 'total_count': 25, 'global_p95': 0.0, 'global_p05': 0.0}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-train-service.duration_mean', 'total_count': 25, 'global_p95': 0.009114507224334576, 'global_p05': 0.00350692532771528}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-train-service.duration_p95', 'total_count': 25, 'global_p95': 0.0547776399999998, 'global_p05': 0.004198409999999921}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-train-service.error_rate', 'total_count': 25, 'global_p95': 0.0, 'global_p05': 0.0}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-train-service.row_count', 'total_count': 25, 'global_p95': 292.4, 'global_p05': 235.4}, {'cmdb_id': 'ts-price-service', 'trace_name': 'trace.from_ts-basic-service.duration_mean', 'total_count': 25, 'global_p95': 0.007914422510421416, 'global_p05': 0.0040611240625}, {'cmdb_id': 'ts-price-service', 'trace_name': 'trace.from_ts-basic-service.duration_p95', 'total_count': 25, 'global_p95': 0.016928899999999962, 'global_p05': 0.00462181}, {'cmdb_id': 'ts-price-service', 'trace_name': 'trace.from_ts-basic-service.error_rate', 'total_count': 25, 'global_p95': 0.0, 'global_p05': 0.0}, {'cmdb_id': 'ts-price-service', 'trace_name': 'trace.from_ts-basic-service.row_count', 'total_count': 25, 'global_p95': 147.39999999999998, 'global_p05': 110.6}, {'cmdb_id': 'ts-price-service', 'trace_name': 'trace.to_ts-price-service.duration_mean', 'total_count': 25, 'global_p95': 0.002069010948905099, 'global_p05': 0.00137024518518518}, {'cmdb_id': 'ts-price-service', 'trace_name': 'trace.to_ts-price-service.duration_p95', 'total_count': 25, 'global_p95': 0.003076879999999979, 'global_p05': 0.0018624799999999}, {'cmdb_id': 'ts-price-service', 'trace_name': 'trace.to_ts-price-service.error_rate', 'total_count': 25, 'global_p95': 0.0, 'global_p05': 0.0}, {'cmdb_id': 'ts-price-service', 'trace_name': 'trace.to_ts-price-service.row_count', 'total_count': 25, 'global_p95': 314.79999999999995, 'global_p05': 242.0}, {'cmdb_id': 'ts-route-service', 'trace_name': 'trace.from_ts-basic-service.duration_mean', 'total_count': 25, 'global_p95': 0.04196722535826739, 'global_p05': 0.0036557861334353603}, {'cmdb_id': 'ts-route-service', 'trace_name': 'trace.from_ts-basic-service.duration_p95', 'total_count': 25, 'global_p95': 0.21064144, 'global_p05': 0.00397989999999998}, {'cmdb_id': 'ts-route-service', 'trace_name': 'trace.from_ts-basic-service.error_rate', 'total_count': 25, 'global_p95': 0.0, 'global_p05': 0.0}, {'cmdb_id': 'ts-route-service', 'trace_name': 'trace.from_ts-basic-service.row_count', 'total_count': 25, 'global_p95': 147.39999999999998, 'global_p05': 110.6}, {'cmdb_id': 'ts-price-service', 'trace_name': 'trace.from_ts-price-service.duration_mean', 'total_count': 25, 'global_p95': 0.002069010948905099, 'global_p05': 0.00137024518518518}, {'cmdb_id': 'ts-price-service', 'trace_name': 'trace.from_ts-price-service.duration_p95', 'total_count': 25, 'global_p95': 0.003076879999999979, 'global_p05': 0.0018624799999999}, {'cmdb_id': 'ts-price-service', 'trace_name': 'trace.from_ts-price-service.error_rate', 'total_count': 25, 'global_p95': 0.0, 'global_p05': 0.0}, {'cmdb_id': 'ts-price-service', 'trace_name': 'trace.from_ts-price-service.row_count', 'total_count': 25, 'global_p95': 314.79999999999995, 'global_p05': 242.0}, {'cmdb_id': 'root', 'trace_name': 'trace.to_ts-travel2-service.duration_mean', 'total_count': 25, 'global_p95': 0.6967223316563573, 'global_p05': 0.31614731323360185}, {'cmdb_id': 'root', 'trace_name': 'trace.to_ts-travel2-service.duration_p95', 'total_count': 25, 'global_p95': 1.576144159999999, 'global_p05': 0.42456782}, {'cmdb_id': 'root', 'trace_name': 'trace.to_ts-travel2-service.error_rate', 'total_count': 25, 'global_p95': 0.0, 'global_p05': 0.0}, {'cmdb_id': 'root', 'trace_name': 'trace.to_ts-travel2-service.row_count', 'total_count': 25, 'global_p95': 78.6, 'global_p05': 69.2}, {'cmdb_id': 'ts-route-service', 'trace_name': 'trace.from_ts-route-service.duration_mean', 'total_count': 25, 'global_p95': 0.027575722170192295, 'global_p05': 0.00124297433416794}, {'cmdb_id': 'ts-route-service', 'trace_name': 'trace.from_ts-route-service.duration_p95', 'total_count': 25, 'global_p95': 0.20745441999999992, 'global_p05': 0.0016763099999999001}, {'cmdb_id': 'ts-route-service', 'trace_name': 'trace.from_ts-route-service.error_rate', 'total_count': 25, 'global_p95': 0.0, 'global_p05': 0.0}, {'cmdb_id': 'ts-route-service', 'trace_name': 'trace.from_ts-route-service.row_count', 'total_count': 25, 'global_p95': 2095.6, 'global_p05': 1782.4}, {'cmdb_id': 'ts-route-service', 'trace_name': 'trace.from_ts-travel2-service.duration_mean', 'total_count': 25, 'global_p95': 0.02985234640397067, 'global_p05': 0.00373109139109882}, {'cmdb_id': 'ts-route-service', 'trace_name': 'trace.from_ts-travel2-service.duration_p95', 'total_count': 25, 'global_p95': 0.2096894, 'global_p05': 0.00432343999999992}, {'cmdb_id': 'ts-route-service', 'trace_name': 'trace.from_ts-travel2-service.error_rate', 'total_count': 25, 'global_p95': 0.0, 'global_p05': 0.0}, {'cmdb_id': 'ts-route-service', 'trace_name': 'trace.from_ts-travel2-service.row_count', 'total_count': 25, 'global_p95': 598.8, 'global_p05': 478.0}, {'cmdb_id': 'ts-route-service', 'trace_name': 'trace.to_ts-route-service.duration_mean', 'total_count': 25, 'global_p95': 0.027575722170192295, 'global_p05': 0.00124297433416794}, {'cmdb_id': 'ts-route-service', 'trace_name': 'trace.to_ts-route-service.duration_p95', 'total_count': 25, 'global_p95': 0.20745441999999992, 'global_p05': 0.0016763099999999001}, {'cmdb_id': 'ts-route-service', 'trace_name': 'trace.to_ts-route-service.error_rate', 'total_count': 25, 'global_p95': 0.0, 'global_p05': 0.0}, {'cmdb_id': 'ts-route-service', 'trace_name': 'trace.to_ts-route-service.row_count', 'total_count': 25, 'global_p95': 2095.6, 'global_p05': 1782.4}, {'cmdb_id': 'ts-seat-service', 'trace_name': 'trace.from_ts-travel2-service.duration_mean', 'total_count': 25, 'global_p95': 0.10818752763157893, 'global_p05': 0.044178178301347425}, {'cmdb_id': 'ts-seat-service', 'trace_name': 'trace.from_ts-travel2-service.duration_p95', 'total_count': 25, 'global_p95': 0.3063975999999999, 'global_p05': 0.06010661999999984}, {'cmdb_id': 'ts-seat-service', 'trace_name': 'trace.from_ts-travel2-service.error_rate', 'total_count': 25, 'global_p95': 0.0, 'global_p05': 0.0}, {'cmdb_id': 'ts-seat-service', 'trace_name': 'trace.from_ts-travel2-service.row_count', 'total_count': 25, 'global_p95': 161.6, 'global_p05': 125.4}, {'cmdb_id': 'ts-seat-service', 'trace_name': 'trace.from_ts-seat-service.duration_mean', 'total_count': 25, 'global_p95': 0.02340319013151148, 'global_p05': 0.0105328601299273}, {'cmdb_id': 'ts-seat-service', 'trace_name': 'trace.from_ts-seat-service.duration_p95', 'total_count': 25, 'global_p95': 0.08537639999999999, 'global_p05': 0.01971407999999998}], 'trace_anomaly_summary': [{'cmdb_id': 'ts-basic-service', 'trace_name': 'trace.to_ts-route-service.row_count', 'total_count': 25, 'global_p95': 147.39999999999998, 'global_p05': 110.6, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101980, 'max_value_in_window': 160.0, 'min_value_in_window': 2.0, 'severity': 55.3}, {'cmdb_id': 'ts-basic-service', 'trace_name': 'trace.to_ts-train-service.row_count', 'total_count': 25, 'global_p95': 147.39999999999998, 'global_p05': 110.6, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101980, 'max_value_in_window': 160.0, 'min_value_in_window': 2.0, 'severity': 55.3}, {'cmdb_id': 'ts-train-service', 'trace_name': 'trace.from_ts-basic-service.row_count', 'total_count': 25, 'global_p95': 147.39999999999998, 'global_p05': 110.6, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101980, 'max_value_in_window': 160.0, 'min_value_in_window': 2.0, 'severity': 55.3}, {'cmdb_id': 'ts-route-service', 'trace_name': 'trace.from_ts-basic-service.row_count', 'total_count': 25, 'global_p95': 147.39999999999998, 'global_p05': 110.6, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101980, 'max_value_in_window': 160.0, 'min_value_in_window': 2.0, 'severity': 55.3}, {'cmdb_id': 'ts-train-service', 'trace_name': 'trace.to_ts-train-service.row_count', 'total_count': 25, 'global_p95': 2644.7999999999993, 'global_p05': 2132.0, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101980, 'max_value_in_window': 2848.0, 'min_value_in_window': 40.0, 'severity': 53.3}, {'cmdb_id': 'ts-train-service', 'trace_name': 'trace.from_ts-train-service.row_count', 'total_count': 25, 'global_p95': 2644.7999999999993, 'global_p05': 2132.0, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101980, 'max_value_in_window': 2848.0, 'min_value_in_window': 40.0, 'severity': 53.3}, {'cmdb_id': 'ts-route-service', 'trace_name': 'trace.from_ts-route-service.row_count', 'total_count': 25, 'global_p95': 2095.6, 'global_p05': 1782.4, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101980, 'max_value_in_window': 2222.0, 'min_value_in_window': 36.0, 'severity': 49.51111111111111}, {'cmdb_id': 'ts-route-service', 'trace_name': 'trace.to_ts-route-service.row_count', 'total_count': 25, 'global_p95': 2095.6, 'global_p05': 1782.4, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101980, 'max_value_in_window': 2222.0, 'min_value_in_window': 36.0, 'severity': 49.51111111111111}, {'cmdb_id': 'root', 'trace_name': 'trace.to_ts-travel2-service.row_count', 'total_count': 25, 'global_p95': 78.6, 'global_p05': 69.2, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680, 'max_value_in_window': 97.0, 'min_value_in_window': 2.0, 'severity': 34.6}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.from_root.row_count', 'total_count': 25, 'global_p95': 78.6, 'global_p05': 69.2, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680, 'max_value_in_window': 97.0, 'min_value_in_window': 2.0, 'severity': 34.6}, {'cmdb_id': 'ts-route-service', 'trace_name': 'trace.from_ts-travel2-service.row_count', 'total_count': 25, 'global_p95': 598.8, 'global_p05': 478.0, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680, 'max_value_in_window': 605.0, 'min_value_in_window': 16.0, 'severity': 29.875}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-route-service.row_count', 'total_count': 25, 'global_p95': 598.8, 'global_p05': 478.0, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680, 'max_value_in_window': 605.0, 'min_value_in_window': 16.0, 'severity': 29.875}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-train-service.row_count', 'total_count': 25, 'global_p95': 292.4, 'global_p05': 235.4, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680, 'max_value_in_window': 295.0, 'min_value_in_window': 8.0, 'severity': 29.425}, {'cmdb_id': 'ts-train-service', 'trace_name': 'trace.from_ts-travel2-service.row_count', 'total_count': 25, 'global_p95': 292.4, 'global_p05': 235.4, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680, 'max_value_in_window': 295.0, 'min_value_in_window': 8.0, 'severity': 29.425}, {'cmdb_id': 'ts-auth-service', 'trace_name': 'trace.to_ts-auth-service.row_count', 'total_count': 25, 'global_p95': 342.0, 'global_p05': 260.40000000000003, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680, 'max_value_in_window': 344.0, 'min_value_in_window': 12.0, 'severity': 21.700000000000003}, {'cmdb_id': 'ts-auth-service', 'trace_name': 'trace.from_ts-auth-service.row_count', 'total_count': 25, 'global_p95': 342.0, 'global_p05': 260.40000000000003, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680, 'max_value_in_window': 344.0, 'min_value_in_window': 12.0, 'severity': 21.700000000000003}, {'cmdb_id': 'ts-auth-service', 'trace_name': 'trace.from_root.row_count', 'total_count': 25, 'global_p95': 85.6, 'global_p05': 65.2, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680, 'max_value_in_window': 87.0, 'min_value_in_window': 4.0, 'severity': 16.3}, {'cmdb_id': 'root', 'trace_name': 'trace.to_ts-auth-service.row_count', 'total_count': 25, 'global_p95': 85.6, 'global_p05': 65.2, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680, 'max_value_in_window': 87.0, 'min_value_in_window': 4.0, 'severity': 16.3}, {'cmdb_id': 'ts-order-service', 'trace_name': 'trace.to_ts-order-service.duration_p95', 'total_count': 24, 'global_p95': 0.010223352499999892, 'global_p05': 0.0032466374999999, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680, 'max_value_in_window': 0.0352105999999998, 'min_value_in_window': 0.0031290999999999, 'severity': 3.4441343972048477}, {'cmdb_id': 'ts-order-service', 'trace_name': 'trace.from_ts-order-service.duration_p95', 'total_count': 24, 'global_p95': 0.010223352499999892, 'global_p05': 0.0032466374999999, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680, 'max_value_in_window': 0.0352105999999998, 'min_value_in_window': 0.0031290999999999, 'severity': 3.4441343972048477}], 'summary_counts': {'total_trace_pairs_scanned': 328, 'anomalous_pairs': 20}}```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse log_df from previous state
df = log_df.copy()

candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# 1) Compute global P95/P05 and total_count per (cmdb_id, log_name) using full series
log_thresholds_all = (
    df.groupby(["cmdb_id", "log_name"])["value"]
    .agg(total_count="size",
         global_p95=lambda s: float(s.quantile(0.95)),
         global_p05=lambda s: float(s.quantile(0.05)))
    .reset_index()
)

# Keep only candidate services
log_thresholds = log_thresholds_all[log_thresholds_all["cmdb_id"].isin(candidates)].reset_index(drop=True)

# 2) Filter to incident window (UTC)
df["ts_utc"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
start = pd.to_datetime("2024-01-24 13:05:01", utc=True)
end   = pd.to_datetime("2024-01-24 13:35:01", utc=True)
window_df = df[(df["ts_utc"] >= start) & (df["ts_utc"] <= end)].copy()

# Keep only candidate services in the window
window_df = window_df[window_df["cmdb_id"].isin(candidates)].copy()

# Total log pairs scanned for the candidates (from thresholds)
total_log_pairs_scanned = int(log_thresholds.shape[0])

# Prepare anomaly summary
if not window_df.empty and not log_thresholds.empty:
    # Merge thresholds into window rows
    window_with_thresh = window_df.merge(log_thresholds, on=["cmdb_id", "log_name"], how="left", validate="m:1")

    # Mark anomalies using global thresholds
    window_with_thresh["is_high_anom"] = window_with_thresh["value"] >= window_with_thresh["global_p95"]
    window_with_thresh["is_low_anom"]  = window_with_thresh["value"] <= window_with_thresh["global_p05"]
    window_with_thresh["is_anom"] = window_with_thresh["is_high_anom"] | window_with_thresh["is_low_anom"]

    # Group and summarize per pair
    def summarize(g):
        anom_count = int(g["is_anom"].sum())
        earliest = int(g.loc[g["is_anom"], "timestamp"].min()) if anom_count > 0 else None
        return pd.Series({
            "anomaly_count_in_window": anom_count,
            "earliest_anomaly_ts": earliest,
            "max_value_in_window": float(g["value"].max()) if not g["value"].isna().all() else None,
            "min_value_in_window": float(g["value"].min()) if not g["value"].isna().all() else None,
        })

    grp = window_with_thresh.groupby(["cmdb_id", "log_name"]).apply(summarize).reset_index()

    # Merge with thresholds to include global_p95/p05 and total_count
    log_anom = grp.merge(log_thresholds, left_on=["cmdb_id", "log_name"], right_on=["cmdb_id", "log_name"], how="left", validate="1:1")

    # Compute severity per rules
    def compute_severity(row):
        gp95 = row["global_p95"]
        gp05 = row["global_p05"]
        max_w = row["max_value_in_window"]
        min_w = row["min_value_in_window"]

        sev_high = -np.inf
        sev_low = -np.inf

        # high-side severity
        if gp95 is not None and not pd.isna(gp95):
            if gp95 != 0 and max_w is not None:
                sev_high = float(max_w / gp95)
            elif gp95 == 0 and max_w is not None and max_w > 0:
                sev_high = np.inf

        # low-side severity
        if gp05 is not None and not pd.isna(gp05):
            if (min_w is not None) and (min_w != 0):
                sev_low = float(gp05 / min_w)
            elif (min_w is not None) and (min_w == 0) and gp05 > 0:
                sev_low = np.inf

        if sev_high == -np.inf and sev_low == -np.inf:
            return np.nan
        return float(max(sev_high, sev_low))

    log_anom["severity"] = log_anom.apply(compute_severity, axis=1)

    # Keep only pairs with anomaly_count_in_window > 0
    log_anom = log_anom[log_anom["anomaly_count_in_window"] > 0].copy()

    # Select and order requested columns
    log_anom = log_anom[[
        "cmdb_id", "log_name", "total_count", "global_p95", "global_p05",
        "anomaly_count_in_window", "earliest_anomaly_ts", "max_value_in_window", "min_value_in_window", "severity"
    ]]

    # Sort by severity desc and limit top 20
    log_anom = log_anom.sort_values("severity", ascending=False).head(20).reset_index(drop=True)

    # Ensure JSON-serializable native types
    int_cols = ["total_count", "anomaly_count_in_window", "earliest_anomaly_ts"]
    for c in int_cols:
        if c in log_anom:
            log_anom[c] = log_anom[c].apply(lambda x: int(x) if pd.notna(x) else None)
    float_cols = ["global_p95", "global_p05", "max_value_in_window", "min_value_in_window", "severity"]
    for c in float_cols:
        if c in log_anom:
            log_anom[c] = log_anom[c].apply(lambda x: float(x) if pd.notna(x) else None)

    anomalous_pairs = int(log_anom.shape[0])
else:
    # No data or no thresholds
    log_anom = pd.DataFrame(columns=[
        "cmdb_id", "log_name", "total_count", "global_p95", "global_p05",
        "anomaly_count_in_window", "earliest_anomaly_ts", "max_value_in_window", "min_value_in_window", "severity"
    ])
    anomalous_pairs = 0

# Prepare thresholds output for candidates only
log_thresholds_out = log_thresholds[["cmdb_id", "log_name", "total_count", "global_p95", "global_p05"]].copy()
log_thresholds_out["total_count"] = log_thresholds_out["total_count"].astype(int)
log_thresholds_out["global_p95"] = log_thresholds_out["global_p95"].astype(float)
log_thresholds_out["global_p05"] = log_thresholds_out["global_p05"].astype(float)

# Final result dict (compact)
result = {
    "log_thresholds": log_thresholds_out.to_dict(orient="records"),
    "log_anomaly_summary": log_anom.to_dict(orient="records"),
    "summary_counts": {
        "total_log_pairs_scanned": total_log_pairs_scanned,
        "anomalous_pairs": anomalous_pairs
    }
}

result

```
Out[4]:
```
import pandas as pd

def ts(u):
    return pd.to_datetime(int(u), unit="s", utc=True).strftime("%Y-%m-%d %H:%M:%S %Z")

summary = (
    "Log analysis summary (plain English):\n"
    "- Scope: candidate services = [ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service]; "
    "incident window = 2024-01-24 13:05:01 to 2024-01-24 13:35:01 UTC.\n"
    f"- Total (cmdb_id, log_name) pairs scanned for candidates: 8. Pairs flagged as anomalous in the window: 8.\n\n"
    "- Key global thresholds (P95) for log.row_count (computed on full series):\n"
    "  * ts-route-service: global_p95 = 1047.8\n"
    "  * ts-auth-service:  global_p95 = 342.0\n"
    "  * ts-travel-service: global_p95 = 2108.6\n"
    "  * ts-order-service: global_p95 = 470.0\n\n"
    "- Top log anomalies (by severity):\n"
    f"  1) ts-route-service / log.row_count — severity=49.51; max_in_window=1111.0 vs global_p95=1047.8; earliest_anomaly={ts(1706101980)}\n"
    f"  2) ts-auth-service / log.row_count — severity=21.70; max_in_window=344.0 vs global_p95=342.0; earliest_anomaly={ts(1706101680)}\n"
    f"  3) ts-travel-service / log.row_count — severity=1.24; max_in_window=2359.0 vs global_p95=2108.6; earliest_anomaly={ts(1706101980)}\n"
    f"  4) ts-order-service / log.row_count — severity=1.23; max_in_window=534.0 vs global_p95=470.0; earliest_anomaly={ts(1706101680)}\n\n"
    "- Note on log.error_count: global P95/P05 = 0 for error_count metrics; all observed values are 0 in the window, so anomaly_count_in_window is nonzero but severity is not meaningful (division by zero). These zero error_count records are not informative as anomalies.\n\n"
    "Bottom line: the most meaningful log signals are spikes in log.row_count during the incident window, especially ts-route-service and ts-auth-service (largest severities). Investigate increased request/row counts for those services and correlated trace metrics."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

{'log_thresholds': [{'cmdb_id': 'ts-auth-service', 'log_name': 'log.error_count', 'total_count': 25, 'global_p95': 0.0, 'global_p05': 0.0}, {'cmdb_id': 'ts-auth-service', 'log_name': 'log.row_count', 'total_count': 25, 'global_p95': 342.0, 'global_p05': 260.40000000000003}, {'cmdb_id': 'ts-order-service', 'log_name': 'log.error_count', 'total_count': 24, 'global_p95': 0.0, 'global_p05': 0.0}, {'cmdb_id': 'ts-order-service', 'log_name': 'log.row_count', 'total_count': 24, 'global_p95': 470.0, 'global_p05': 253.9}, {'cmdb_id': 'ts-route-service', 'log_name': 'log.error_count', 'total_count': 25, 'global_p95': 0.0, 'global_p05': 0.0}, {'cmdb_id': 'ts-route-service', 'log_name': 'log.row_count', 'total_count': 25, 'global_p95': 1047.8, 'global_p05': 891.2}, {'cmdb_id': 'ts-travel-service', 'log_name': 'log.error_count', 'total_count': 24, 'global_p95': 0.0, 'global_p05': 0.0}, {'cmdb_id': 'ts-travel-service', 'log_name': 'log.row_count', 'total_count': 24, 'global_p95': 2108.6, 'global_p05': 1210.95}], 'log_anomaly_summary': [{'cmdb_id': 'ts-route-service', 'log_name': 'log.row_count', 'total_count': 25, 'global_p95': 1047.8, 'global_p05': 891.2, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101980, 'max_value_in_window': 1111.0, 'min_value_in_window': 18.0, 'severity': 49.51111111111111}, {'cmdb_id': 'ts-auth-service', 'log_name': 'log.row_count', 'total_count': 25, 'global_p95': 342.0, 'global_p05': 260.40000000000003, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101680, 'max_value_in_window': 344.0, 'min_value_in_window': 12.0, 'severity': 21.700000000000003}, {'cmdb_id': 'ts-travel-service', 'log_name': 'log.row_count', 'total_count': 24, 'global_p95': 2108.6, 'global_p05': 1210.95, 'anomaly_count_in_window': 4, 'earliest_anomaly_ts': 1706101980, 'max_value_in_window': 2359.0, 'min_value_in_window': 978.0, 'severity': 1.2381901840490799}, {'cmdb_id': 'ts-order-service', 'log_name': 'log.row_count', 'total_count': 24, 'global_p95': 470.0, 'global_p05': 253.9, 'anomaly_count_in_window': 5, 'earliest_anomaly_ts': 1706101680, 'max_value_in_window': 534.0, 'min_value_in_window': 206.0, 'severity': 1.2325242718446603}, {'cmdb_id': 'ts-auth-service', 'log_name': 'log.error_count', 'total_count': 25, 'global_p95': 0.0, 'global_p05': 0.0, 'anomaly_count_in_window': 25, 'earliest_anomaly_ts': 1706101680, 'max_value_in_window': 0.0, 'min_value_in_window': 0.0, 'severity': nan}, {'cmdb_id': 'ts-order-service', 'log_name': 'log.error_count', 'total_count': 24, 'global_p95': 0.0, 'global_p05': 0.0, 'anomaly_count_in_window': 24, 'earliest_anomaly_ts': 1706101680, 'max_value_in_window': 0.0, 'min_value_in_window': 0.0, 'severity': nan}, {'cmdb_id': 'ts-route-service', 'log_name': 'log.error_count', 'total_count': 25, 'global_p95': 0.0, 'global_p05': 0.0, 'anomaly_count_in_window': 25, 'earliest_anomaly_ts': 1706101680, 'max_value_in_window': 0.0, 'min_value_in_window': 0.0, 'severity': nan}, {'cmdb_id': 'ts-travel-service', 'log_name': 'log.error_count', 'total_count': 24, 'global_p95': 0.0, 'global_p05': 0.0, 'anomaly_count_in_window': 24, 'earliest_anomaly_ts': 1706101680, 'max_value_in_window': 0.0, 'min_value_in_window': 0.0, 'severity': nan}], 'summary_counts': {'total_log_pairs_scanned': 8, 'anomalous_pairs': 8}}```
```